# **Redefinitions of Units of Measure**

This notebook provides a comprehensive framework for testing the ability of Large Language Models (LLMs) to answer questions about redefining units of measure across three difficulty levels. The generated responses are saved in .csv files for further analysis.

In [ ]:
!pip install --upgrade boto3 
!pip install --upgrade botocore
!pip install langchain

We begin by accessing a selected LLM model via AWS Bedrock. To establish access, you must provide your AWS access key credentials.

In [ ]:
# Import required libraries
import boto3
import json
from botocore.exceptions import ClientError
from langchain.prompts import PromptTemplate

# AWS Bedrock setup
client = boto3.client(
    service_name="bedrock-runtime",
    aws_access_key_id='your_aws_access_key_id',
    aws_secret_access_key='your_aws_secret_access_key',
    region_name="us-west-2"
)

# Model ID for Bedrock
model_id = "amazon.titan-text-lite-v1"

SYSTEM_PROMPT = "You are a helpful assistant."

In [ ]:
import json
import time
import random
from botocore.exceptions import ClientError  


# Invokes the model on AWS Bedrock and handles errors with retries.
# Depending on the selected model, slight adjustments may be needed.

def get_model_response(input_text, retries=5, backoff_factor=2, max_delay=60):
    native_request = {
        "inputText": input_text,
        "textGenerationConfig": {
            "maxTokenCount": 512,
            "temperature": 0.5,
            "topP": 0.9
        }
    }

    request = json.dumps(native_request)

    for attempt in range(1, retries + 1):
        try:
            response = client.invoke_model(modelId=model_id, body=request)
            model_response = json.loads(response["body"].read())

            # Extract the correct response
            output_text = model_response["results"][0]["outputText"]
            return output_text

        except (KeyError, IndexError):
            print(f"KeyError on attempt {attempt}: {model_response}")
        except ClientError as e:
            print(f"ClientError on attempt {attempt}: {e}")
        except Exception as e:
            print(f"Error on attempt {attempt}: {e}")
        
        if attempt == retries:
            print("Max retries reached. Returning None.")
            return None

        delay = min(backoff_factor * (2 ** (attempt - 1)), max_delay) + random.uniform(0, 1)
        print(f"Retrying in {delay:.2f} seconds...")
        time.sleep(delay)

In [ ]:
def get_model_answer(prompt_template, question, retries=5):
    input_text = prompt_template.format(question=question)
    return get_model_response(input_text, retries=retries)

In [ ]:
def fix_answers(llm_answers):
    llm_answers_fixed = []
    for answer in llm_answers:
        if answer is not None and 'final answer is: ' in answer:
            llm_answers_fixed.append(answer.split('final answer is: ', 1)[1])
        else:
            llm_answers_fixed.append(answer)  
    return llm_answers_fixed

# **Datasets**

The datasets used in these experiments contain information about the relationships between different units of measure, their redefined relationships, and the corresponding questions along with their answers related to these relationships. There are six datasets for the redefinition of constants: three in free-form (FF) format and three in multiple-choice (MC) format, each corresponding to a different difficulty level.

In [2]:
import os
import pandas as pd

is_kaggle = os.path.exists('/kaggle/input')

# If you're running this notebook in the Kaggle environment you need to manually upload the files contained in the 'input' folder
#   Upload these files to the Kaggle "Input" section before running the code.
#   Make sure the folder structure matches the expected directories:
#   - 'units-level1' folder containing 'units of measure - level1.csv' and 'units of measure - level1_mc.csv'
#   - 'units-level2' folder containing 'units of measure - level2.csv' and 'units of measure - level2_mc.csv'
#   - 'units-level3' folder containing 'units of measure - level3.csv' and 'units of measure - level3_mc.csv'

if is_kaggle:
    input_dir = '/kaggle/input'
else:
    input_dir = os.path.join(os.getcwd(), 'input')

level1_dir = os.path.join(input_dir, "units-level1")
level2_dir = os.path.join(input_dir, "units-level2")
level3_dir = os.path.join(input_dir, "units-level3")

file_level1 = ["units of measure - level1.csv", "units of measure - level1_mc.csv"]
file_level2 = ["units of measure - level2.csv", "units of measure - level2_mc.csv"]
file_level3 = ["units of measure - level3.csv", "units of measure - level3_mc.csv"]

df1 = pd.read_csv(os.path.join(level1_dir, file_level1[0]))  
df1_mc = pd.read_csv(os.path.join(level1_dir, file_level1[1]))  

df2 = pd.read_csv(os.path.join(level2_dir, file_level2[0]))  
df2_mc = pd.read_csv(os.path.join(level2_dir, file_level2[1])) 

df3 = pd.read_csv(os.path.join(level3_dir, file_level3[0]))  
df3_mc = pd.read_csv(os.path.join(level3_dir, file_level3[1]))  

In [5]:
data_level1 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"]
    }
    for _, row in df1.iterrows()
]
data_level1[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'How many seconds are in 2 minutes?',
  'answer_real': '120',
  'Y1': '100 seconds',
  'answer1': '200',
  'Y2': '5×10^8 seconds',
  'answer2': '10^9',
  'Y3': '-50 seconds',
  'answer3': '-100'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': 'How many grams are in 2 kilograms?',
  'answer_real': '2000',
  'Y1': '900 grams',
  'answer1': '1800',
  'Y2': '10^(-14) grams',
  'answer2': '2×10^(-14)',
  'Y3': '-100 grams',
  'answer3': '-200'}]

In [6]:
data_level2 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"]
    }
    for _, row in df2.iterrows()
]
data_level2[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'A stopwatch runs for 3 and a half minutes. How many seconds does it count?',
  'answer_real': '210',
  'Y1': '100 seconds',
  'answer1': '350',
  'Y2': '5×10^8 seconds',
  'answer2': '1.75×10^9',
  'Y3': '-50 seconds',
  'answer3': '-175'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': 'A person weighs 72 kilograms. What is the persons weight in grams?',
  'answer_real': '72000',
  'Y1': '900 grams',
  'answer1': '64800',
  'Y2': '10^(-14) grams',
  'answer2': '7.2×10^(-13)',
  'Y3': '-100 grams',
  'answer3': '-7200'}]

In [7]:
data_level3 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"]
    }
    for _, row in df3.iterrows()
]
data_level3[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'A marathon runner runs at a speed of 170 meters per minute. How many seconds will it take them to complete a 42-kilometer race?',
  'answer_real': '14,823.53',
  'Y1': '100 seconds',
  'answer1': '24,705.8824',
  'Y2': '5×10^8 seconds',
  'answer2': '1.235×10^11',
  'Y3': '-50 seconds',
  'answer3': '-12,352.94'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': "A vehicle's engine weighs 650 kilograms. If 15% of the weight is aluminum, what is the weight of the aluminum in grams?",
  'answer_real': '97,500 grams',
  'Y1': '900 grams',
  'answer1': '87,750 grams',
  'Y2': '10^(-14) grams',
  'answer2': '9.75×10^(−13) grams',
  'Y3': '-100 grams',
  'answer3': '-9,750 grams'}]

In [8]:
data_level1_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "A2": row["A2"],
        "B2": row["B2"],
        "C2": row["C2"],
        "D2": row["D2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "A3": row["A3"],
        "B3": row["B3"],
        "C3": row["C3"],
        "D3": row["D3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"]
    }
    for _, row in df1_mc.iterrows()
]
data_level1_mc[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'How many seconds are in 2 minutes?',
  'answer_real': '120',
  'Y1': '100 seconds',
  'A1': '100',
  'B1': '120',
  'C1': '200',
  'D1': '220',
  'answer1': 'C',
  'answer1_real': 'B',
  'Y2': '5×10^8 seconds',
  'A2': '120',
  'B2': '10^8',
  'C2': '5×10^8',
  'D2': '10^9',
  'answer2': 'D',
  'answer2_real': 'A',
  'Y3': '-50 seconds',
  'A3': '-120',
  'B3': '120',
  'C3': '-100',
  'D3': '50',
  'answer3': 'C',
  'answer3_real': 'B'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': 'How many grams are in 2 kilograms?',
  'answer_real': '2000',
  'Y1': '900 grams',
  'A1': '1000',
  'B1': '1500',
  'C1': '1800',
  'D1': '2000',
  'answer1': 'C',
  'answer1_real': 'D',
  'Y2': '10^(-14) grams',
  'A2': '10^(-14)',
  'B2': '2×10^(-14)',
  'C2': '1000',
  'D2': '2000',
  'answer2': 'B',
  'answer2_real': 'D',
  'Y3': '-100 grams',
  'A3': '-200',
  'B3': '200',
  'C3': '1000',
  'D3': '2000',
  'answer3': 

In [9]:
data_level2_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "A2": row["A2"],
        "B2": row["B2"],
        "C2": row["C2"],
        "D2": row["D2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "A3": row["A3"],
        "B3": row["B3"],
        "C3": row["C3"],
        "D3": row["D3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"]
    }
    for _, row in df2_mc.iterrows()
]
data_level2_mc[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'A stopwatch runs for 3 and a half minutes. How many seconds does it count?',
  'answer_real': '210',
  'Y1': '100 seconds',
  'A1': '180',
  'B1': '210',
  'C1': '300',
  'D1': '350',
  'answer1': 'D',
  'answer1_real': 'B',
  'Y2': '5×10^8 seconds',
  'A2': '210',
  'B2': '350',
  'C2': '1.75×10^9',
  'D2': '2.1×10^9',
  'answer2': 'C',
  'answer2_real': 'A',
  'Y3': '-50 seconds',
  'A3': '-175',
  'B3': '175',
  'C3': '210',
  'D3': '-150',
  'answer3': 'A',
  'answer3_real': 'C'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': 'A person weighs 72 kilograms. What is the persons weight in grams?',
  'answer_real': '72000',
  'Y1': '900 grams',
  'A1': '47300',
  'B1': '56000',
  'C1': '64800',
  'D1': '72000',
  'answer1': 'C',
  'answer1_real': 'D',
  'Y2': '10^(-14) grams',
  'A2': '720',
  'B2': '72000',
  'C2': '7.2×10^(10)',
  'D2': '7.2×10^(-13)',
  'answer2': 'D',
  'answer2_real': 'B',
  'Y3': '

In [10]:
data_level3_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "A1": row["A1"],
        "B1": row["B1"],
        "C1": row["C1"],
        "D1": row["D1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "A2": row["A2"],
        "B2": row["B2"],
        "C2": row["C2"],
        "D2": row["D2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "A3": row["A3"],
        "B3": row["B3"],
        "C3": row["C3"],
        "D3": row["D3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"]
    }
    for _, row in df3_mc.iterrows()
]
data_level3_mc[:2]

[{'X': '1 minute',
  'real_value': '60 seconds',
  'question': 'A marathon runner runs at a speed of 170 meters per minute. How many seconds will it take them to complete a 42-kilometer race?',
  'answer_real': '14,823.53',
  'Y1': '100 seconds',
  'A1': '9,235.76',
  'B1': '14,823.53',
  'C1': '24,705.8824',
  'D1': '56,287.43',
  'answer1': 'C',
  'answer1_real': 'B',
  'Y2': '5×10^8 seconds',
  'A2': '14,823.53',
  'B2': '12,537,000',
  'C2': '1.14×10^10',
  'D2': '1.235×10^11',
  'answer2': 'D',
  'answer2_real': 'A',
  'Y3': '-50 seconds',
  'A3': '-12,352.94',
  'B3': '13,574.64',
  'C3': '14,823.53',
  'D3': '-15,935.68',
  'answer3': 'A',
  'answer3_real': 'C'},
 {'X': '1 kilogram',
  'real_value': '1000 grams',
  'question': "A vehicle's engine weighs 650 kilograms. If 15% of the weight is aluminum, what is the weight of the aluminum in grams?",
  'answer_real': '97,500 grams',
  'Y1': '900 grams',
  'A1': '77,265 grams',
  'B1': '87,750 grams',
  'C1': '97,500 grams',
  'D1':

# **No Redefinition**

We first ask these questions without redefining the relationships between units of measure to assess the model's baseline knowledge. In this notebook, for the no-redefinition phase, we focus on the free-form format using zero-shot prompting. Other experiments exploring different prompting techniques for both free-form and multiple-choice formats can be found in no_redefinition.ipynb.

In [ ]:
template = """
Answer the following question:
{question}

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

prompt_template = PromptTemplate(
    input_variables=["question"],
    template=template
)

In [ ]:
answers1 = []

for row in data_level1:
    question = row['question']
    answers1.append(get_model_answer(prompt_template, question))

In [ ]:
answers2 = []

for row in data_level2:
    question = row['question']
    answers2.append(get_model_answer(prompt_template, question))

In [ ]:
answers3 = []

for row in data_level3:
    question = row['question']
    answers3.append(get_model_answer(prompt_template, question))

In [ ]:
answers1_fixed = fix_answers(answers1)
answers2_fixed = fix_answers(answers2)
answers3_fixed = fix_answers(answers3)

# **Redefinition**

We conduct experiments for both free-form and multiple-choice formats varying by redefinition levels, question difficulty levels, and prompting techniques. 

## **Free Form**

### **Zero-Shot**

In [ ]:
template_redefinition = """
Redefine {X} as {Y}. {question}

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition = PromptTemplate(
    input_variables=["question", "X", "Y"],
    template=template_redefinition
)

In [ ]:
def get_model_answer_redefinition(prompt_template, question, X, Y, retries=5):
    input_text = prompt_template.format(question=question, X=X, Y=Y)
    return get_model_response(input_text, retries=retries)

In [ ]:
answers1_Y1_redefinition, answers1_Y2_redefinition, answers1_Y3_redefinition = [], [], []
answers2_Y1_redefinition, answers2_Y2_redefinition, answers2_Y3_redefinition = [], [], []
answers3_Y1_redefinition, answers3_Y2_redefinition, answers3_Y3_redefinition = [], [], []

In [ ]:
for row in data_level1:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    
    answers1_Y1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y1))
    answers1_Y2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y2))
    answers1_Y3_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y3))

In [ ]:
for row in data_level2:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
 
    answers2_Y1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y1))
    answers2_Y2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y2))
    answers2_Y3_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y3))

In [ ]:
for row in data_level3:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    
    answers3_Y1_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y1))
    answers3_Y2_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y2))
    answers3_Y3_redefinition.append(get_model_answer_redefinition(prompt_template_redefinition, question, X , Y3))

In [ ]:
answers1_Y1_redefinition_fixed = fix_answers(answers1_Y1_redefinition)
answers1_Y2_redefinition_fixed = fix_answers(answers1_Y2_redefinition)
answers1_Y3_redefinition_fixed = fix_answers(answers1_Y3_redefinition)

In [ ]:
answers2_Y1_redefinition_fixed = fix_answers(answers2_Y1_redefinition)
answers2_Y2_redefinition_fixed = fix_answers(answers2_Y2_redefinition)
answers2_Y3_redefinition_fixed = fix_answers(answers2_Y3_redefinition)

In [ ]:
answers3_Y1_redefinition_fixed = fix_answers(answers3_Y1_redefinition)
answers3_Y2_redefinition_fixed = fix_answers(answers3_Y2_redefinition)
answers3_Y3_redefinition_fixed = fix_answers(answers3_Y3_redefinition)

### **Chain-of-Thought (CoT)**

In [ ]:
template_redefinition_cot = """
Redefine {X} as {Y}. {question}

Let's think step by step. 

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition_cot = PromptTemplate(
    input_variables=["question", "X", "Y"],
    template=template_redefinition_cot
)

In [ ]:
answers1_Y1_redefinition_cot, answers1_Y2_redefinition_cot, answers1_Y3_redefinition_cot = [], [], []
answers2_Y1_redefinition_cot, answers2_Y2_redefinition_cot, answers2_Y3_redefinition_cot = [], [], []
answers3_Y1_redefinition_cot, answers3_Y2_redefinition_cot, answers3_Y3_redefinition_cot = [], [], []

In [ ]:
for row in data_level1:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    
    answers1_Y1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y1))
    answers1_Y2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y2))
    answers1_Y3_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y3))

In [ ]:
for row in data_level2:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    
    answers2_Y1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y1))
    answers2_Y2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y2))
    answers2_Y3_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y3))

In [ ]:
for row in data_level3:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    
    answers3_Y1_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y1))
    answers3_Y2_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y2))
    answers3_Y3_redefinition_cot.append(get_model_answer_redefinition(prompt_template_redefinition_cot, question, X , Y3))

In [ ]:
answers1_Y1_redefinition_cot_fixed = fix_answers(answers1_Y1_redefinition_cot)
answers1_Y2_redefinition_cot_fixed = fix_answers(answers1_Y2_redefinition_cot)
answers1_Y3_redefinition_cot_fixed = fix_answers(answers1_Y3_redefinition_cot)

In [ ]:
answers2_Y1_redefinition_cot_fixed = fix_answers(answers2_Y1_redefinition_cot)
answers2_Y2_redefinition_cot_fixed = fix_answers(answers2_Y2_redefinition_cot)
answers2_Y3_redefinition_cot_fixed = fix_answers(answers2_Y3_redefinition_cot)

In [ ]:
answers3_Y1_redefinition_cot_fixed = fix_answers(answers3_Y1_redefinition_cot)
answers3_Y2_redefinition_cot_fixed = fix_answers(answers3_Y2_redefinition_cot)
answers3_Y3_redefinition_cot_fixed = fix_answers(answers3_Y3_redefinition_cot)

### **Few-Shot**

In [ ]:
template_redefinition_f = """
Redefine {X} as {Y}. {question}

Here are some examples of similar questions with their correct answers:

Question: Redefine 1 day as 10 hours. How many hours are in 2 days?
Answer: 20

Question: Redefine 1 degree as 0.5 radians. How many radians are in a full circle?
Answer: 180

Question: Redefine 1 ohm as 200 milliohms. A lightbulb has a resistance of 50 ohms. What is its resistance in milliohms?
Answer: 10,000

End the response with the phrase "The final answer is: " followed only by the correct result, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition_f = PromptTemplate(
    input_variables=["question", "X", "Y"],
    template=template_redefinition_f
)

In [ ]:
answers1_Y1_redefinition_f, answers1_Y2_redefinition_f, answers1_Y3_redefinition_f = [], [], []
answers2_Y1_redefinition_f, answers2_Y2_redefinition_f, answers2_Y3_redefinition_f = [], [], []
answers3_Y1_redefinition_f, answers3_Y2_redefinition_f, answers3_Y3_redefinition_f = [], [], []

In [ ]:
for row in data_level1:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    
    answers1_Y1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y1))
    answers1_Y2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y2))
    answers1_Y3_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y3))

In [ ]:
for row in data_level2:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    
    answers2_Y1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y1))
    answers2_Y2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y2))
    answers2_Y3_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y3))

In [ ]:
for row in data_level3:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    Y2 = row['Y2']
    Y3 = row['Y3']
    
    answers3_Y1_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y1))
    answers3_Y2_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y2))
    answers3_Y3_redefinition_f.append(get_model_answer_redefinition(prompt_template_redefinition_f, question, X, Y3))

In [ ]:
answers1_Y1_redefinition_f_fixed = fix_answers(answers1_Y1_redefinition_f)
answers1_Y2_redefinition_f_fixed = fix_answers(answers1_Y2_redefinition_f)
answers1_Y3_redefinition_f_fixed = fix_answers(answers1_Y3_redefinition_f)

In [ ]:
answers2_Y1_redefinition_f_fixed = fix_answers(answers2_Y1_redefinition_f)
answers2_Y2_redefinition_f_fixed = fix_answers(answers2_Y2_redefinition_f)
answers2_Y3_redefinition_f_fixed = fix_answers(answers2_Y3_redefinition_f)

In [ ]:
answers3_Y1_redefinition_f_fixed = fix_answers(answers3_Y1_redefinition_f)
answers3_Y2_redefinition_f_fixed = fix_answers(answers3_Y2_redefinition_f)
answers3_Y3_redefinition_f_fixed = fix_answers(answers3_Y3_redefinition_f)

## **Multiple Choice**

### **Zero-Shot**

In [ ]:
template_redefinition_mc = """
Redefine {X} as {Y}. Choose A, B, C, or D to answer the question.

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition_mc = PromptTemplate(
    input_variables=["question", "X", "Y", "A", "B", "C", "D"],
    template=template_redefinition_mc
)

In [ ]:
def get_model_answer_redefinition_mc(prompt_template, question, X, Y, A, B, C, D, retries=5):
    input_text = prompt_template.format(question=question, X=X, Y=Y, A=A, B=B, C=C, D=D)
    return get_model_response(input_text, retries=retries)

In [ ]:
answers1_Y1_redefinition_mc, answers1_Y2_redefinition_mc, answers1_Y3_redefinition_mc = [], [], []
answers2_Y1_redefinition_mc, answers2_Y2_redefinition_mc, answers2_Y3_redefinition_mc = [], [], []
answers3_Y1_redefinition_mc, answers3_Y2_redefinition_mc, answers3_Y3_redefinition_mc = [], [], []

In [ ]:
for row in data_level1_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers1_Y1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y1, A1, B1, C1, D1))
    answers1_Y2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y2, A2, B2, C2, D2))
    answers1_Y3_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y3, A3, B3, C3, D3))

In [ ]:
for row in data_level2_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers2_Y1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y1, A1, B1, C1, D1))
    answers2_Y2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y2, A2, B2, C2, D2))
    answers2_Y3_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y3, A3, B3, C3, D3))

In [ ]:
for row in data_level3_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers3_Y1_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y1, A1, B1, C1, D1))
    answers3_Y2_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y2, A2, B2, C2, D2))
    answers3_Y3_redefinition_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc, question, X, Y3, A3, B3, C3, D3))

In [ ]:
answers1_Y1_redefinition_mc_fixed = fix_answers(answers1_Y1_redefinition_mc)
answers1_Y2_redefinition_mc_fixed = fix_answers(answers1_Y2_redefinition_mc)
answers1_Y3_redefinition_mc_fixed = fix_answers(answers1_Y3_redefinition_mc)

In [ ]:
answers2_Y1_redefinition_mc_fixed = fix_answers(answers2_Y1_redefinition_mc)
answers2_Y2_redefinition_mc_fixed = fix_answers(answers2_Y2_redefinition_mc)
answers2_Y3_redefinition_mc_fixed = fix_answers(answers2_Y3_redefinition_mc)

In [ ]:
answers3_Y1_redefinition_mc_fixed = fix_answers(answers3_Y1_redefinition_mc)
answers3_Y2_redefinition_mc_fixed = fix_answers(answers3_Y2_redefinition_mc)
answers3_Y3_redefinition_mc_fixed = fix_answers(answers3_Y3_redefinition_mc)

### **Chain-of-Thought (CoT)**

In [ ]:
template_redefinition_mc_cot = """
Redefine {X} as {Y}. Choose A, B, C, or D to answer the question.

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Let's think step by step. 

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

prompt_template_redefinition_mc_cot = PromptTemplate(
    input_variables=["question", "X", "Y", "A", "B", "C", "D"],
    template=template_redefinition_mc_cot
)

In [ ]:
answers1_Y1_redefinition_cot_mc, answers1_Y2_redefinition_cot_mc, answers1_Y3_redefinition_cot_mc = [], [], []
answers2_Y1_redefinition_cot_mc, answers2_Y2_redefinition_cot_mc, answers2_Y3_redefinition_cot_mc = [], [], []
answers3_Y1_redefinition_cot_mc, answers3_Y2_redefinition_cot_mc, answers3_Y3_redefinition_cot_mc = [], [], []

In [ ]:
for row in data_level1_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers1_Y1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y1, A1, B1, C1, D1))
    answers1_Y2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y2, A2, B2, C2, D2))
    answers1_Y3_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y3, A3, B3, C3, D3))

In [ ]:
for row in data_level2_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers2_Y1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y1, A1, B1, C1, D1))
    answers2_Y2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y2, A2, B2, C2, D2))
    answers2_Y3_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y3, A3, B3, C3, D3))

In [ ]:
for row in data_level3_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers3_Y1_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y1, A1, B1, C1, D1))
    answers3_Y2_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y2, A2, B2, C2, D2))
    answers3_Y3_redefinition_cot_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_cot, question, X, Y3, A3, B3, C3, D3))

In [ ]:
answers1_Y1_redefinition_cot_mc_fixed = fix_answers(answers1_Y1_redefinition_cot_mc)
answers1_Y2_redefinition_cot_mc_fixed = fix_answers(answers1_Y2_redefinition_cot_mc)
answers1_Y3_redefinition_cot_mc_fixed = fix_answers(answers1_Y3_redefinition_cot_mc)

In [ ]:
answers2_Y1_redefinition_cot_mc_fixed = fix_answers(answers2_Y1_redefinition_cot_mc)
answers2_Y2_redefinition_cot_mc_fixed = fix_answers(answers2_Y2_redefinition_cot_mc)
answers2_Y3_redefinition_cot_mc_fixed = fix_answers(answers2_Y3_redefinition_cot_mc)

In [ ]:
answers3_Y1_redefinition_cot_mc_fixed = fix_answers(answers3_Y1_redefinition_cot_mc)
answers3_Y2_redefinition_cot_mc_fixed = fix_answers(answers3_Y2_redefinition_cot_mc)
answers3_Y3_redefinition_cot_mc_fixed = fix_answers(answers3_Y3_redefinition_cot_mc)

### **Few-Shot**

In [ ]:
template_redefinition_mc_f = """
Redefine {X} as {Y}. Choose A, B, C, or D to answer the question.

Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}

Here are some examples of similar questions with their correct answers:

Question: Redefine 1 day as 10 hours. How many hours are in 2 days?
A: 10
B: 20
C: 24
D: 48
Answer: B

Question: Redefine 1 degree as 0.5 radians. How many radians are in a full circle?
A: pi
B: 2*pi
C: 180
D: 360
Answer: C

Question: Redefine 1 ohm as 200 milliohms. A lightbulb has a resistance of 50 ohms. What is its resistance in milliohms?
A: 5,000 milliohms
B: 10,000 milliohms
C: 50,000 milliohms
D: 100,000 milliohms
Answer: B

Provide only the letter corresponding to the correct answer: "A", "B", "C", or "D".
End the response with the phrase "The final answer is: " followed by the correct letter, with no additional text or commentary.
"""

from langchain.prompts import PromptTemplate

prompt_template_redefinition_mc_f = PromptTemplate(
    input_variables=["question", "X", "Y", "A", "B", "C", "D"],
    template=template_redefinition_mc_f
)

In [ ]:
answers1_Y1_redefinition_f_mc, answers1_Y2_redefinition_f_mc, answers1_Y3_redefinition_f_mc = [], [], []
answers2_Y1_redefinition_f_mc, answers2_Y2_redefinition_f_mc, answers2_Y3_redefinition_f_mc = [], [], []
answers3_Y1_redefinition_f_mc, answers3_Y2_redefinition_f_mc, answers3_Y3_redefinition_f_mc = [], [], []

In [ ]:
for row in data_level1_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers1_Y1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y1, A1, B1, C1, D1))
    answers1_Y2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y2, A2, B2, C2, D2))
    answers1_Y3_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y3, A3, B3, C3, D3))

In [ ]:
for row in data_level2_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers2_Y1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y1, A1, B1, C1, D1))
    answers2_Y2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y2, A2, B2, C2, D2))
    answers2_Y3_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y3, A3, B3, C3, D3))

In [ ]:
for row in data_level3_mc:
    question = row['question']
    X = row['X']
    Y1 = row['Y1']
    A1, B1, C1, D1 = row['A1'], row['B1'], row['C1'], row['D1']
    Y2 = row['Y2']
    A2, B2, C2, D2 = row['A2'], row['B2'], row['C2'], row['D2']
    Y3 = row['Y3']
    A3, B3, C3, D3 = row['A3'], row['B3'], row['C3'], row['D3']

    answers3_Y1_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y1, A1, B1, C1, D1))
    answers3_Y2_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y2, A2, B2, C2, D2))
    answers3_Y3_redefinition_f_mc.append(get_model_answer_redefinition_mc(prompt_template_redefinition_mc_f, question, X, Y3, A3, B3, C3, D3))

In [ ]:
answers1_Y1_redefinition_f_mc_fixed = fix_answers(answers1_Y1_redefinition_f_mc)
answers1_Y2_redefinition_f_mc_fixed = fix_answers(answers1_Y2_redefinition_f_mc)
answers1_Y3_redefinition_f_mc_fixed = fix_answers(answers1_Y3_redefinition_f_mc)

In [ ]:
answers2_Y1_redefinition_f_mc_fixed = fix_answers(answers2_Y1_redefinition_f_mc)
answers2_Y2_redefinition_f_mc_fixed = fix_answers(answers2_Y2_redefinition_f_mc)
answers2_Y3_redefinition_f_mc_fixed = fix_answers(answers2_Y3_redefinition_f_mc)

In [ ]:
answers3_Y1_redefinition_f_mc_fixed = fix_answers(answers3_Y1_redefinition_f_mc)
answers3_Y2_redefinition_f_mc_fixed = fix_answers(answers3_Y2_redefinition_f_mc)
answers3_Y3_redefinition_f_mc_fixed = fix_answers(answers3_Y3_redefinition_f_mc)

# **Results**

We store all generated responses in .csv files for further analysis.

In [ ]:
data_model_level1 = {
    "no_redefinition": answers1_fixed,
    "Y1_redefinition_zero": answers1_Y1_redefinition_fixed,
    "Y1_redefinition_cot": answers1_Y1_redefinition_cot_fixed,
    "Y1_redefinition_few": answers1_Y1_redefinition_f_fixed,
    
    "Y2_redefinition_zero": answers1_Y2_redefinition_fixed,
    "Y2_redefinition_cot": answers1_Y2_redefinition_cot_fixed,
    "Y2_redefinition_few": answers1_Y2_redefinition_f_fixed,
    
    "Y3_redefinition_zero": answers1_Y3_redefinition_fixed,
    "Y3_redefinition_cot": answers1_Y3_redefinition_cot_fixed,
    "Y3_redefinition_few": answers1_Y3_redefinition_f_fixed,

    
    "Y1_redefinition_mc_zero": answers1_Y1_redefinition_mc_fixed,
    "Y1_redefinition_mc_cot": answers1_Y1_redefinition_cot_mc_fixed,
    "Y1_redefinition_mc_few": answers1_Y1_redefinition_f_mc_fixed,
    
    "Y2_redefinition_mc_zero": answers1_Y2_redefinition_mc_fixed,
    "Y2_redefinition_mc_cot": answers1_Y2_redefinition_cot_mc_fixed,
    "Y2_redefinition_mc_few": answers1_Y2_redefinition_f_mc_fixed,
    
    "Y3_redefinition_mc_zero": answers1_Y3_redefinition_mc_fixed,
    "Y3_redefinition_mc_cot": answers1_Y3_redefinition_cot_mc_fixed,
    "Y3_redefinition_mc_few": answers1_Y3_redefinition_f_mc_fixed

}

df_answers1 = pd.DataFrame(data_model_level1)

df_answers1.head()

In [ ]:
data_model_level2 = {
    "no_redefinition": answers2_fixed,
    "Y1_redefinition_zero": answers2_Y1_redefinition_fixed,
    "Y1_redefinition_cot": answers2_Y1_redefinition_cot_fixed,
    "Y1_redefinition_few": answers2_Y1_redefinition_f_fixed,
    
    "Y2_redefinition_zero": answers2_Y2_redefinition_fixed,
    "Y2_redefinition_cot": answers2_Y2_redefinition_cot_fixed,
    "Y2_redefinition_few": answers2_Y2_redefinition_f_fixed,
    
    "Y3_redefinition_zero": answers2_Y3_redefinition_fixed,
    "Y3_redefinition_cot": answers2_Y3_redefinition_cot_fixed,
    "Y3_redefinition_few": answers2_Y3_redefinition_f_fixed,
    
    
    "Y1_redefinition_mc_zero": answers2_Y1_redefinition_mc_fixed,
    "Y1_redefinition_mc_cot": answers2_Y1_redefinition_cot_mc_fixed,
    "Y1_redefinition_mc_few": answers2_Y1_redefinition_f_mc_fixed,
    
    "Y2_redefinition_mc_zero": answers2_Y2_redefinition_mc_fixed,
    "Y2_redefinition_mc_cot": answers2_Y2_redefinition_cot_mc_fixed,
    "Y2_redefinition_mc_few": answers2_Y2_redefinition_f_mc_fixed,
    
    "Y3_redefinition_mc_zero": answers2_Y3_redefinition_mc_fixed,
    "Y3_redefinition_mc_cot": answers2_Y3_redefinition_cot_mc_fixed,
    "Y3_redefinition_mc_few": answers2_Y3_redefinition_f_mc_fixed

}

df_answers2 = pd.DataFrame(data_model_level2)

df_answers2.head()

In [ ]:
data_model_level3 = {
    "no_redefinition": answers3_fixed,
    "Y1_redefinition_zero": answers3_Y1_redefinition_fixed,
    "Y1_redefinition_cot": answers3_Y1_redefinition_cot_fixed,
    "Y1_redefinition_few": answers3_Y1_redefinition_f_fixed,
    
    "Y2_redefinition_zero": answers3_Y2_redefinition_fixed,
    "Y2_redefinition_cot": answers3_Y2_redefinition_cot_fixed,
    "Y2_redefinition_few": answers3_Y2_redefinition_f_fixed,
    
    "Y3_redefinition_zero": answers3_Y3_redefinition_fixed,
    "Y3_redefinition_cot": answers3_Y3_redefinition_cot_fixed,
    "Y3_redefinition_few": answers3_Y3_redefinition_f_fixed,
    
    
    "Y1_redefinition_mc_zero": answers3_Y1_redefinition_mc_fixed,
    "Y1_redefinition_mc_cot": answers3_Y1_redefinition_cot_mc_fixed,
    "Y1_redefinition_mc_few": answers3_Y1_redefinition_f_mc_fixed,
    
    "Y2_redefinition_mc_zero": answers3_Y2_redefinition_mc_fixed,
    "Y2_redefinition_mc_cot": answers3_Y2_redefinition_cot_mc_fixed,
    "Y2_redefinition_mc_few": answers3_Y2_redefinition_f_mc_fixed,
    
    "Y3_redefinition_mc_zero": answers3_Y3_redefinition_mc_fixed,
    "Y3_redefinition_mc_cot": answers3_Y3_redefinition_cot_mc_fixed,
    "Y3_redefinition_mc_few": answers3_Y3_redefinition_f_mc_fixed
    
}

df_answers3 = pd.DataFrame(data_model_level3)

df_answers3.head()

In [ ]:
df_answers1.to_csv("llm_results1.csv", index=False)
df_answers2.to_csv("llm_results2.csv", index=False)
df_answers3.to_csv("llm_results3.csv", index=False)